# MyTravels Infrastructure Runbook — Helm

This notebook is the Helm-based equivalent of `runbook.ipynb`. Instead of applying each manifest under `manifests/` by hand, it installs the chart at `helm/mytravels`, which renders and applies the same set of resources in one shot. Read `runbook.ipynb` first if you want the manifest-by-manifest walkthrough — this notebook assumes that context and focuses on the Helm workflow.

## Summary

- **Step 1 — Prerequisites**: Verify the required tools are installed (Rancher Desktop, k3d, kubectl, Helm, JupyterLab, OpenLens).
- **Step 2 — Create the Cluster**: Create the local k3d cluster (1 control plane, 3 workers) with port mappings for Traefik HTTP (8080) and PostgreSQL TCP (5432).
- **Step 3 — /etc/hosts**: Add `*.mytravels.local` hostnames to `/etc/hosts` for host-based ingress routing.
- **Step 4 — Review Chart Values**: Inspect `helm/mytravels/values.yaml` and override anything that isn't safe to ship as-is (e.g. the Content Safety placeholders).
- **Step 5 — Install the Chart**: `helm install` the full stack — namespace, PostgreSQL, RabbitMQ, MinIO, the `db-migrations` Job, API, Messaging, the Traefik `postgres` entrypoint, ingress rules, and Backstage RBAC — in a single command.
- **Step 6 — Database Migrations**: Wait for the once-off `db-migrations` Job to complete.
- **Step 7 — Traefik Postgres Entrypoint**: Confirm the chart's `HelmChartConfig` patch took effect.
- **Step 8 — Full Stack Verification**: Confirm all pods, PVCs, services, ingresses, and URLs are healthy.
- **Step 9 — Backstage RBAC**: Print the ServiceAccount token for the Backstage Kubernetes plugin.
- **Step 10 — Diagnostics**: Pull logs and events per service when something misbehaves.
- **Step 11 — Upgrading**: How to `helm upgrade` safely, including the `db-migrations` Job's immutability gotcha.
- **Step 12 — Teardown**: `helm uninstall` and optionally delete the whole cluster.

## The architecture

![architecture](images/architecture.png)

---

## Step 1 — Prerequisites

Rancher Desktop, k3d, kubectl, JupyterLab, and Freelens/OpenLens install notes: [macOS](<../1-install tools (macos).md>) · [Ubuntu](<../1-install tools (ubuntu).md>) · [Windows](<../1-install tools (windows).md>).

You'll also need Helm 3 — see [helm.sh/docs/intro/install](https://helm.sh/docs/intro/install/).

Once Rancher Desktop is installed, open it and ensure the container engine is running before continuing.

In [ ]:
%%bash
echo "=== Docker ==="
docker --version
echo "=== k3d ==="
k3d --version
echo "=== kubectl ==="
kubectl version --client 2>/dev/null || kubectl version --client --short
echo "=== helm ==="
helm version --short

---

## Step 2 — Create the Cluster

![cluster](images/k8s%20components.drawio.png)

Creates a local k3d cluster with 1 control plane node and 3 worker nodes. Traefik is bundled automatically by k3s and serves as the ingress controller — the chart's `traefikPostgres` template patches it the same way `manifests/8-traefik-config.yaml` does.

| Flag | Meaning |
|---|---|
| `-p "8080:80@loadbalancer"` | Maps `localhost:8080` → cluster port 80 (Traefik web entrypoint) |
| `-p "5432:5432@loadbalancer"` | Maps `localhost:5432` → cluster port 5432 (Traefik postgres TCP entrypoint) |
| `--image ghcr.io/k3s-io/k3s:v1.35.3-k3s1` | Pins the k3s version for reproducible cluster creation |
| `--servers 1` | 1 control plane node |
| `--agents 3` | 3 worker nodes |

> Skip this cell if the cluster already exists (`k3d cluster list`).

> **Ensure Rancher Desktop is running before this cell.** k3d creates the cluster's nodes as Docker containers, so it needs a live Docker daemon — on Linux there's no system Docker install, Rancher Desktop *is* the daemon. If it isn't running (or hasn't finished starting its VM yet), `k3d cluster create` fails immediately with `Cannot connect to the Docker daemon at unix:///home/<user>/.rd/docker.sock`, because that socket file doesn't exist until Rancher Desktop creates it. Open Rancher Desktop and wait for it to fully start, then confirm with `docker info` before retrying.

> **MinIO note:** the chart's default `minio.nodeHostname` is `k3d-mytravels-agent-2`, which only exists if the cluster was created with `--agents 3` (or more). If you use fewer agents, override that value (Step 4) before installing.

In [ ]:
%%bash
k3d cluster create mytravels \
  -p "8080:80@loadbalancer" \
  -p "5432:5432@loadbalancer" \
  --image ghcr.io/k3s-io/k3s:v1.35.3-k3s1 \
  --servers 1 \
  --agents 3

In [ ]:
%%bash
echo "=== Nodes ==="
kubectl get nodes
echo ""
echo "=== Traefik ==="
kubectl get pods -n kube-system -l app.kubernetes.io/name=traefik

---

## Step 3 — /etc/hosts

The chart's `ingress.yaml` template uses host-based routing, defaulting to the same hostnames as `manifests/9-ingress.yaml`. Add the entries below to your hosts file so your browser resolves them to localhost. If you override `ingress.*Host` in `values.yaml` (Step 4), update these to match.

```
127.0.0.1  rabbitmq.mytravels.local
127.0.0.1  minio.mytravels.local
127.0.0.1  api.mytravels.local
127.0.0.1  messaging.mytravels.local
```

Run **one** of the next two cells depending on your OS:

- **macOS/Linux** — appends to `/etc/hosts` via `sudo`, prompting for your password.
- **Windows** — appends to `C:\Windows\System32\drivers\etc\hosts`. There's no `sudo` on Windows, so instead the cell checks whether it has Administrator privileges and writes directly if so. If not, close Jupyter/VS Code and relaunch it "as Administrator", then re-run the cell.

### macOS/Linux

In [ ]:
import subprocess
import getpass

password = getpass.getpass("sudo password: ")

hosts = ["rabbitmq.mytravels.local", "minio.mytravels.local", "api.mytravels.local", "messaging.mytravels.local"]

with open("/etc/hosts", "r") as f:
    current = f.read()

for host in hosts:
    if host in current:
        print(f"Already present: {host}")
    else:
        entry = f"127.0.0.1  {host}\n"
        result = subprocess.run(
            ["sudo", "-S", "tee", "-a", "/etc/hosts"],
            input=f"{password}\n{entry}",
            capture_output=True,
            text=True
        )
        if result.returncode == 0:
            print(f"Added: {host}")
        else:
            print(f"Failed: {host} — {result.stderr.strip()}")

### Windows

In [ ]:
import ctypes

hosts_path = r"C:\Windows\System32\drivers\etc\hosts"
hosts = ["rabbitmq.mytravels.local", "minio.mytravels.local", "api.mytravels.local", "messaging.mytravels.local"]

def is_admin():
    try:
        return bool(ctypes.windll.shell32.IsUserAnAdmin())
    except Exception:
        return False

if not is_admin():
    print("Not running as Administrator — the hosts file is not writable.")
    print("Close Jupyter/VS Code and relaunch it via 'Run as Administrator', then re-run this cell.")
else:
    with open(hosts_path, "r") as f:
        current = f.read()

    with open(hosts_path, "a") as f:
        for host in hosts:
            if host in current:
                print(f"Already present: {host}")
            else:
                f.write(f"127.0.0.1  {host}\n")
                print(f"Added: {host}")

---

## Step 4 — Review Chart Values

`helm/mytravels/values.yaml` holds everything that's split across the individual `1-secret.yaml` files and hardcoded env vars in `manifests/`: image tags, resource requests/limits, storage classes, ingress hosts, and the dummy tutorial credentials (`user123` / `password123`) that match `manifests/*/1-secret.yaml` exactly.

One thing is **not** safe to install as-is: `messaging.contentSafetyEndpoint` and `messaging.contentSafetyKey` default to `CHANGE_ME` placeholders (the real values in `manifests/messaging/1-secret.yaml` are Azure Content Safety credentials — deliberately not copied into the chart). Override them at install time rather than editing `values.yaml` in place, so real secrets don't end up committed:

```bash
helm install mytravels ./helm/mytravels \
  --namespace mytravels-default --create-namespace \
  --set-string messaging.contentSafetyEndpoint="https://your-endpoint.cognitiveservices.azure.com/" \
  --set-string messaging.contentSafetyKey="your-key"
```

Or put overrides in a gitignored `my-values.yaml` and pass `-f my-values.yaml` instead. The cell below just renders the chart locally (no cluster changes) so you can review exactly what will be applied.

In [ ]:
%%bash
echo "=== Default values ==="
helm show values ./helm/mytravels
echo ""
echo "=== Rendered manifests (dry run, no cluster contact) ==="
helm template mytravels ./helm/mytravels --namespace mytravels-default | head -50
echo "... (truncated — see full output with: helm template mytravels ./helm/mytravels)" 

---

## Step 5 — Install the Chart

This single command replaces Steps 4–14 of `runbook.ipynb`: it creates the `mytravels-default` namespace, then deploys PostgreSQL, RabbitMQ, MinIO, the `db-migrations` Job, the API, the Messaging worker, the Traefik `postgres` TCP entrypoint patch, the ingress rules, and the Backstage RBAC resources.

Skip the `--set-string` flags below if you already added your Content Safety values via a values file in Step 4.

In [ ]:
%%bash
helm install mytravels ./helm/mytravels \
  --namespace mytravels-default --create-namespace

In [ ]:
%%bash
helm status mytravels -n mytravels-default
echo ""
helm get manifest mytravels -n mytravels-default | grep -E '^kind:' | sort | uniq -c

---

## Step 6 — Database Migrations

The chart's `db-migrations` Job behaves identically to `manifests/migrations/2-job.yaml`: the `cleanup-migrations` initContainer loops on `pg_isready` until PostgreSQL is reachable, deletes specific rows from `EFMigrationsHistory`, then `migrate-core-db` runs `dotnet ef database update`. Because it's a plain Job (not a Helm hook), it's created as part of Step 5 above but its own readiness loop is what actually gates it on PostgreSQL being up — it isn't gated on the API/Messaging deployments, which may start and crash-loop until the schema is ready. That's expected; Kubernetes restarts them automatically once migrations finish.

In [ ]:
%%bash
# Poll until the pod is scheduled (handles the case where the cell runs before the pod exists)
until kubectl get pod -l job-name=db-migrations -n mytravels-default 2>/dev/null | grep -q db-migrations; do
  echo "Waiting for pod to be scheduled..."; sleep 2
done

# Wait until the init container finishes (pod moves past PodInitializing)
kubectl wait pod -l job-name=db-migrations -n mytravels-default \
  --for=condition=Initialized --timeout=120s

# Wait for the job to complete — guarantees migrate-core-db has started and exited
kubectl wait job/db-migrations -n mytravels-default \
  --for=condition=Complete --timeout=300s

echo "--LIST JOBS--"
kubectl get job db-migrations -n mytravels-default
echo "--CLEANUP MIGRATION LOGS--"
kubectl logs -n mytravels-default -l job-name=db-migrations -c cleanup-migrations
echo "--MIGRATION LOGS--"
kubectl logs -n mytravels-default -l job-name=db-migrations -c migrate-core-db

---

## Step 7 — Traefik Postgres Entrypoint

The chart's `traefik-config.yaml` template (gated by `traefikPostgres.enabled`, default `true`) applies the same `HelmChartConfig/traefik` patch as `manifests/8-traefik-config.yaml`, adding `--entrypoints.postgres.address=:5432/tcp`. k3s watches for the change and restarts Traefik automatically within ~15 seconds — this cell just waits and confirms it landed.

In [ ]:
%%bash
echo "Waiting for Traefik to restart..."
sleep 20
kubectl rollout status deploy/traefik -n kube-system --timeout=60s
echo ""
for i in $(seq 1 12); do
  ARGS=$(kubectl get deploy traefik -n kube-system -o jsonpath='{.spec.template.spec.containers[0].args}' | tr ',' '\n')
  if echo "$ARGS" | grep -q postgres; then
    echo "postgres entrypoint confirmed:"
    echo "$ARGS" | grep postgres
    break
  fi
  echo "Waiting for postgres entrypoint... ($i/12)"
  sleep 5
done

---

## Step 8 — Full Stack Verification

Run these cells to confirm all resources are healthy before using the stack.

In [ ]:
%%bash
echo "=== Pods ==="
kubectl get pods -n mytravels-default
echo ""
echo "=== PVCs ==="
kubectl get pvc -n mytravels-default
echo ""
echo "=== Services ==="
kubectl get svc -n mytravels-default
echo ""
echo "=== Ingress ==="
kubectl get ingress -n mytravels-default

In [ ]:
%%bash
echo "=== RabbitMQ Management ==="
curl -s -o /dev/null -w "%{http_code}" http://rabbitmq.mytravels.local:8080 && echo " OK" || echo " UNREACHABLE"
echo ""
echo "=== MinIO Console ==="
curl -s -o /dev/null -w "%{http_code}" http://minio.mytravels.local:8080 && echo " OK" || echo " UNREACHABLE"
echo ""
echo "=== API ==="
curl -s -o /dev/null -w "%{http_code}" http://api.mytravels.local:8080 && echo " OK" || echo " UNREACHABLE"
echo ""
echo "=== Messaging ==="
curl -s -o /dev/null -w "%{http_code}" http://messaging.mytravels.local:8080/health && echo " OK" || echo " UNREACHABLE"
echo ""

In [ ]:
%%bash
kubectl top pods -A

**Services deployed:**

| Service | Purpose |
|---|---|
| PostgreSQL | Primary database |
| RabbitMQ | Message broker |
| MinIO | S3-compatible object storage |
| API | ASP.NET Core REST API |
| Messaging | ASP.NET Core background worker (RabbitMQ consumer) |

**Management UIs (after full setup):**

| Service | URL |
|---|---|
| RabbitMQ Management | [http://rabbitmq.mytravels.local:8080](http://rabbitmq.mytravels.local:8080) |
| MinIO Console | [http://minio.mytravels.local:8080](http://minio.mytravels.local:8080) |
| API | [http://api.mytravels.local:8080/swagger](http://api.mytravels.local:8080/swagger) |

---

## Step 9 — Backstage RBAC

The chart's `backstage-rbac.yaml` template (gated by `backstageRbac.enabled`, default `true`) creates the same read-only ServiceAccount, ClusterRole, ClusterRoleBinding, and token Secret as `manifests/10-backstage-rbac.yaml`.

> **After Step 5:** run the cell below to print the cluster URL, token, and CA data, then paste those values into the `kubernetes` section of `app-config.local.yaml` in your Backstage instance. The cluster URL changes each time the cluster is recreated — re-run this cell and update the config whenever you rebuild.

In [ ]:
%%bash
# Confirm the ServiceAccount token was issued, then print the values needed
# for app-config.local.yaml in your Backstage instance
kubectl wait secret/backstage-token -n mytravels-default \
  --for=jsonpath='{.data.token}' --timeout=30s
echo ""
echo "=== ServiceAccount ==="
kubectl get serviceaccount backstage -n mytravels-default
echo ""
echo "=== Cluster URL ==="
kubectl config view --minify -o jsonpath='{.clusters[0].cluster.server}'
echo ""
echo ""
echo "=== Service Account Token ==="
kubectl get secret backstage-token -n mytravels-default \
  -o jsonpath='{.data.token}' | base64 -d
echo ""
echo ""
echo "=== CA Data (base64) ==="
kubectl config view --raw --minify \
  -o jsonpath='{.clusters[0].cluster.certificate-authority-data}'
echo ""

---

## Step 10 — Diagnostics

Run these cells when a service is not behaving as expected.

In [ ]:
%%bash
echo "=== PostgreSQL logs ==="
kubectl logs -n mytravels-default -l app=postgres --tail=20

In [ ]:
%%bash
echo "=== RabbitMQ logs ==="
kubectl logs -n mytravels-default -l app=rabbitmq --tail=20

In [ ]:
%%bash
echo "=== MinIO logs ==="
kubectl logs -n mytravels-default -l app=minio --tail=20

In [ ]:
%%bash
echo "=== API logs ==="
kubectl logs -n mytravels-default -l app=api --tail=20

In [ ]:
%%bash
echo "=== Messaging logs ==="
kubectl logs -n mytravels-default -l app=messaging --tail=20

In [ ]:
%%bash
# Recent events — useful for diagnosing scheduling or PVC binding failures
kubectl get events -n mytravels-default --sort-by='.lastTimestamp' | tail -20

---

## Step 11 — Upgrading

After editing `values.yaml` (or a values override file) or bumping an image tag, re-run `helm upgrade`:

```bash
helm upgrade mytravels ./helm/mytravels --namespace mytravels-default
```

One gotcha: the `db-migrations` Job's `spec.template` is immutable, so if it's already present, `helm upgrade` will fail trying to update it in place. Delete it first — same as the commented-out line in `runbook.ipynb`'s migrations cell — and the upgrade recreates it fresh, re-running migrations.

In [ ]:
%%bash
kubectl delete job db-migrations -n mytravels-default --ignore-not-found
helm upgrade mytravels ./helm/mytravels --namespace mytravels-default

---

## Step 12 — Teardown

Uninstalling the release removes every resource Helm created (namespace, Secrets, PVCs, PVs, Deployments, Services, Ingress, the Traefik patch, Backstage RBAC). PostgreSQL and RabbitMQ's PVCs use `local-path` with the default `Delete` reclaim policy, so their data goes with them; MinIO's PVs are `Retain`, so the underlying `hostPath` directories on the node survive until the node itself is gone — which happens next if you delete the whole cluster.

Run the first cell to tear down just the release, or both cells to wipe everything.

In [ ]:
%%bash
helm uninstall mytravels -n mytravels-default
kubectl delete namespace mytravels-default --ignore-not-found

In [ ]:
%%bash
# Delete the entire cluster — removes all Docker containers and volumes
k3d cluster delete mytravels